# 01 — Data Preparation and Data Quality

This notebook loads the official NGCP **Luzon Actual Load** worksheet, converts the 24 source-hour columns into a long hourly time series, assigns the three study periods, and checks whether the data are suitable for modeling.

### What to look for
- Every study date should contain exactly **24 hourly observations**.
- Missing, duplicate, or negative demand values should be zero (or explicitly investigated).
- Period coverage should match **2017–2019**, **2020–2021**, and **2022–2024**.
- The source workbook labels columns as **Hour No. 1–24**. Because the workbook itself does not define clock-time labels, this project reports **NGCP Hour No.** in findings. The internal `hour` field is only a zero-based analysis index (`Hour No. - 1`).

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "src" / "data_preparation.py").is_file():
    project_root = project_root.parent
if not (project_root / "src" / "data_preparation.py").is_file():
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import PERIOD_ORDER
figures_dir = project_root / "figures"
results_dir = project_root / "results"
figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

from src.data_preparation import prepare_dataset, data_quality_report
from src.config import NGCP_DATE_COLUMN, NGCP_HEADER_ROW, NGCP_LUZON_SHEET

input_path = project_root / "data/raw/Hourly Demand per Grid.xlsx"
df = prepare_dataset(
    input_path,
    sheet_name=NGCP_LUZON_SHEET,
    header=NGCP_HEADER_ROW,
    date_col=NGCP_DATE_COLUMN,
    study_only=True,
)
quality = data_quality_report(df)
quality

C:\Users\JAC\Documents\projects\css142p-luzon-demand-simulation\src\data_preparation.py:126: UserWarning: Detected NGCP Hour No. columns 1..24. They are preserved in source_hour_label and mapped to a zero-based analysis index 0..23. The workbook itself does not define clock-time labels, so report NGCP Hour No. (1..24) unless an external source establishes a clock-time convention.
  hour_columns, hour_map = _detect_hour_columns(df, date_col)


,metric,value
0,rows,70128
1,unique_dates,2922
2,complete_24_hour_days,2922
3,missing_demand_values,0
4,duplicate_datetimes,0
5,negative_demand_values,0


In [2]:
coverage = (
    df.groupby("period", observed=True)
      .agg(start_date=("date", "min"), end_date=("date", "max"),
           days=("date", "nunique"), observations=("demand_mw", "size"))
      .reindex(PERIOD_ORDER)
      .reset_index()
)
coverage

,period,start_date,end_date,days,observations
0,Pre-pandemic,2017-01-01,2019-12-31,1095,26280
1,Pandemic,2020-01-01,2021-12-31,731,17544
2,Recovery,2022-01-01,2024-12-31,1096,26304


In [3]:
hour_mapping = pd.DataFrame({
    "NGCP Hour No.": range(1, 25),
    "internal hour index": range(24),
})
hour_mapping

,NGCP Hour No.,internal hour index
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4
5,6,5
6,7,6
7,8,7
8,9,8
9,10,9


**Hour-label convention used in this study:** the workbook says only **“Hour No.”** and labels the 24 demand columns from 1 to 24. We therefore avoid unsupported statements such as “Hour 14 = exactly 1:00 PM.” The model uses a zero-based index internally, but tables and plots intended for interpretation use the original **NGCP Hour No. 1–24**.

In [4]:
df.head(30)

,date,source_hour_label,demand_mw,hour,datetime,year,month,day_of_week,day_type,period
0,2017-01-01,1,5064.0,0,2017-01-01 00:00:00,2017,1,Sunday,Weekend,Pre-pandemic
1,2017-01-01,2,4807.0,1,2017-01-01 01:00:00,2017,1,Sunday,Weekend,Pre-pandemic
2,2017-01-01,3,4609.0,2,2017-01-01 02:00:00,2017,1,Sunday,Weekend,Pre-pandemic
3,2017-01-01,4,4457.0,3,2017-01-01 03:00:00,2017,1,Sunday,Weekend,Pre-pandemic
4,2017-01-01,5,4332.0,4,2017-01-01 04:00:00,2017,1,Sunday,Weekend,Pre-pandemic
5,2017-01-01,6,4268.0,5,2017-01-01 05:00:00,2017,1,Sunday,Weekend,Pre-pandemic
6,2017-01-01,7,4091.0,6,2017-01-01 06:00:00,2017,1,Sunday,Weekend,Pre-pandemic
7,2017-01-01,8,4107.0,7,2017-01-01 07:00:00,2017,1,Sunday,Weekend,Pre-pandemic
8,2017-01-01,9,4270.0,8,2017-01-01 08:00:00,2017,1,Sunday,Weekend,Pre-pandemic
9,2017-01-01,10,4431.0,9,2017-01-01 09:00:00,2017,1,Sunday,Weekend,Pre-pandemic


In [5]:
out = project_root / "data/processed/luzon_hourly_clean.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
quality.to_csv(results_dir / "data_quality_report.csv", index=False)
coverage.to_csv(results_dir / "period_coverage.csv", index=False)
print(f"Saved {len(df):,} hourly observations to {out}")
print(f"Complete days: {df['date'].nunique():,}; expected hourly rows: {df['date'].nunique()*24:,}")

Saved 70,128 hourly observations to C:\Users\JAC\Documents\projects\css142p-luzon-demand-simulation\data\processed\luzon_hourly_clean.csv
Complete days: 2,922; expected hourly rows: 70,128


In [6]:
q = dict(zip(quality["metric"], quality["value"]))
checks_ok = (
    q.get("missing_demand_values", 1) == 0
    and q.get("duplicate_datetimes", 1) == 0
    and q.get("negative_demand_values", 1) == 0
    and q.get("complete_24_hour_days", -1) == q.get("unique_dates", -2)
)
print("DATA QUALITY RESULT:", "PASS" if checks_ok else "REVIEW REQUIRED")
if checks_ok:
    print("All study dates have 24 hourly slots, with no missing, duplicate, or negative demand observations.")

DATA QUALITY RESULT: PASS
All study dates have 24 hourly slots, with no missing, duplicate, or negative demand observations.
